# V2 top-up — H6 + agg diagnostic + Task G matched-budget check

**Purpose of this run:** the previous H6 pass found `naive_no_scorer` beats every learned scorer, but on an UNMATCHED calibration budget (learned modes get 80% of E_eval_ID, naive got 30%). This run adds `naive_matched` (80%, same data as the learned modes) so the comparison is budget-matched. If `naive_matched` still wins on width AND coverage -> the H6 conclusion holds and the paper frame locks (see `PAPER_FRAME.md`).

Phase 1 + Phase 2 already completed. This reuses their outputs (from the previous run's `v2_confirm.zip`, attached as a Kaggle input) and runs only:
- **aggregation diagnostic** — does mean / max / L2 / active-count aggregation give `MVDF_v2` / `correlated` a signal with ≥4 populated quantile bins? (the check for whether H3 is structurally unmeasurable, now on 4 datasets)
- **H6 four-way** — `dense` / `mvdf` / `dense+mvdf` / `correlated` / `random_k` / `pca_k`, split-conformal calibration, α=0.10, ID + shift, paired bootstrap.

**Attach BOTH**: the `uom220001c` data bundle AND the previous run's output (upload `v2_confirm.zip` as a dataset). GPU + Internet on. Run All. ~20-25 min.

## 1. Vendor + deps + locate the prior Phase 1 outputs

In [ ]:
import os, base64, json, sys, subprocess, importlib, glob, shutil, zipfile
FILES=json.loads(base64.b64decode('...').decode())
for rel,c in FILES.items():
    p=os.path.join('mechanistic_conformal',rel); os.makedirs(os.path.dirname(p),exist_ok=True); open(p,'w').write(c)
print('wrote',len(FILES),'files')
for pkg in ['pyyaml','scipy','scikit-learn','statsmodels']:
    n={'pyyaml':'yaml','scikit-learn':'sklearn'}.get(pkg,pkg)
    try: importlib.import_module(n)
    except ImportError: subprocess.run([sys.executable,'-m','pip','install','-q',pkg],check=True)
import torch; print('torch',torch.__version__,'cuda',torch.cuda.is_available())
sys.path.insert(0,'mechanistic_conformal'); os.chdir('mechanistic_conformal')

# find v2_phase1_seed* dirs anywhere under /kaggle/input, or inside a zip there
os.makedirs('runs', exist_ok=True)
found=[]
for root,dirs,files in os.walk('/kaggle/input'):
    for d in dirs:
        if d.startswith('v2_phase1_seed'):
            src=os.path.join(root,d); dst=os.path.join('runs',d)
            if not os.path.exists(dst): shutil.copytree(src,dst); found.append(d)
    for f in files:
        if f.endswith('.zip'):
            try:
                with zipfile.ZipFile(os.path.join(root,f)) as z:
                    names=[n for n in z.namelist() if 'v2_phase1_seed' in n]
                    if names:
                        z.extractall('runs/_unzip'); print('extracted', f)
                        for p in glob.glob('runs/_unzip/**/v2_phase1_seed*', recursive=True):
                            b=os.path.basename(p); dst=os.path.join('runs',b)
                            if os.path.isdir(p) and not os.path.exists(dst):
                                shutil.copytree(p,dst); found.append(b)
            except zipfile.BadZipFile: pass
print('Phase 1 dirs available:', sorted(set(os.path.basename(p) for p in glob.glob('runs/v2_phase1_seed*'))))
assert glob.glob('runs/v2_phase1_seed*/*/mvdf_v2_selection.json'), \
    'No Phase 1 selections found. Attach the previous run output (v2_confirm.zip as a dataset).'
from data.probe_inputs import probe; probe()


## 2. Configure

In [ ]:
import yaml, torch
dev='cuda' if torch.cuda.is_available() else 'cpu'
for cf in ['configs/v2_phase2.yaml','configs/v2_phase3_h6.yaml']:
    c=yaml.safe_load(open(cf)); c['device']=dev; c['strict_fingerprint']=True
    c['datasets']=["ETTh1", "ETTh2", "electricity", "traffic"]
    if 'seeds' in c: c['seeds']=[0, 1, 2]
    yaml.safe_dump(c,open(cf,'w'))
print('configured: device',dev)


## 3. Aggregation diagnostic + H6  (one forecaster/SAE per dataset-seed)

`h6_and_diagnostic.py` trains each deterministic iTransformer + SAE **once** and runs both the aggregation diagnostic and the 6-mode split-conformal H6 from it (the earlier top-up design would have retrained twice). ~20-25 min for 4×3.

In [ ]:
import runpy, sys
sys.argv=['hd','--datasets']+[str(x) for x in ["ETTh1", "ETTh2", "electricity", "traffic"]]+['--seeds']+[str(x) for x in [0, 1, 2]]+['--phase1_root_tmpl','runs/v2_phase1_seed{seed}']
try: runpy.run_module('experiments.h6_and_diagnostic', run_name='__main__')
except SystemExit: pass


## 4. Read the results

In [ ]:
import pandas as pd
diag=pd.read_csv('runs/h6_and_diag/AGG_diag.csv')
print('=== AGG DIAGNOSTIC: bins_populated by condition x agg ===')
print(diag.pivot_table(index='condition',columns='agg',values='bins_populated').round(2).to_string())
print('\n=== per-feature activation rate (median) ===')
print(diag.groupby('condition')['median_per_feat_active'].mean().round(3).to_string())
print('  (MVDF_v2/correlated staying ~2-3 bins under ALL aggs -> H3 structurally unmeasurable)')

h6=pd.read_csv('runs/h6_and_diag/AGG_h6.csv')
print('\n=== H6 coverage sanity (split-conformal -> ~0.90 ID) ===')
print(h6.groupby('mode')[['id_coverage','shift_coverage']].mean().round(3).to_string())
print('\n=== per-seed shift width by mode ===')
print(h6.pivot_table(index=['dataset','seed'],columns='mode',values='shift_width').round(3).to_string())
if 'naive_no_scorer' in set(h6['mode']):
    print('\n=== Task G: parameter-free constant-width CP vs the learned-scorer family ===')
    sh=['naive_matched','naive_no_scorer','dense','mvdf'] if 'naive_matched' in set(h6['mode']) else ['naive_no_scorer','dense','mvdf']
    g=h6[h6['mode'].isin(sh)]
    print(g.groupby('mode')[['id_coverage','id_width','shift_coverage','shift_width']].mean().reindex(sh).round(3).to_string())
    print('  naive_no_scorer = 30% conformalize block only; naive_matched = train U conf (80%, matched to learned modes)')
    print('  DECISION: does naive_matched still beat every learned mode on width AND coverage? -> H6 conclusion holds at matched budget')
    if 'naive_matched' in set(h6['mode']):
        em=h6[h6.dataset=='electricity'].pivot_table(index='mode',columns='seed',values='shift_coverage').reindex(sh).round(3)
        print('\n  electricity shift coverage by seed (cf mvdf 0.827/0.630/0.811):'); print(em.to_string())

t=pd.read_csv('runs/h6_and_diag/AGG_h6_tests.csv')
summ=t.groupby(['comparison','env']).agg(mean_diff=('mean_diff','mean'),p_a_lt_b=('p_a_lt_b','mean'),n_sig=('significant','sum'),n=('significant','size')).round(4)
print('\n=== paired bootstrap by comparison x env (mean over 12 dataset-seed runs) ===')
print(summ.to_string())
def sig(comp,env):
    try: r=summ.loc[(comp,env)]; return f'{r.mean_diff:+.3f} (sig {int(r.n_sig)}/{int(r.n)})'
    except KeyError: return 'n/a'
print('\n=== FOUR-WAY READ ===')
for env in ['ID','shift']:
    print(f'  [{env}] mvdf-dense {sig("mvdf_minus_dense",env)}  mvdf-correlated {sig("mvdf_minus_correlated",env)}  '
          f'mvdf-random_k {sig("mvdf_minus_random_k",env)}  mvdf-pca_k {sig("mvdf_minus_pca_k",env)}  '
          f'pca_k-dense {sig("pca_k_minus_dense",env)}')
print()
print('  mvdf<dense but mvdf~correlated~random_k     -> any k-dim SAE selection helps, not causal content')
print('  mvdf<dense AND mvdf<correlated              -> dose-response/R_j content specifically matters')
print('  mvdf~random_k<dense BUT pca_k~dense         -> SAE dictionary basis specifically (not any compression)')


## 5. Zip

In [ ]:
import shutil
shutil.make_archive('/kaggle/working/v2_h6_topup','zip','runs/h6_and_diag')
print('runs/h6_and_diag -> /kaggle/working/v2_h6_topup.zip')
